# Sionna 0.19 Scene Builder

Portable scene builder using:
- **AWS Terrain Tiles** (elevation-tiles-prod, GeoTIFF, zoom 14 ≈ 2.4 m/px) — no spikes, global coverage, free, anonymous
- **OSM via osmnx** — buildings + roads + water + vegetation
- **Approach from sionna-large-radio-maps** (NVIDIA) adapted for Sionna 0.19 / no drjit dependency

### What this notebook does
1. Downloads AWS elevation tiles for the scene bbox → builds a clean heightmap
2. Downloads OSM buildings → extrudes each building with correct `base_z` from heightmap
3. Builds terrain PLY mesh with real DEM elevation
4. Writes `scene.xml` (Mitsuba 2.1.0) compatible with Sionna 0.19
5. Saves everything to `BASE_DIR/scene/` ready for the main simulation notebook

### Run order
`CELL 0 → CELL 1 → CELL 2 → CELL 3 → CELL 4 → CELL 5 → CELL 6`

In [ ]:
# ============================================================
# CELL 0 — CONFIG  (edit this cell only)
# ============================================================
import os

# ── Scene bbox (WGS84) ────────────────────────────────────────────────────
# Tight bbox: TX + first 1200 RX (915.95 MHz) + 900m margin → 10.1×7.3 km = 74 km²
SCENE_WEST   = -1.260093
SCENE_EAST   = -1.129307
SCENE_SOUTH  =  52.945798
SCENE_NORTH  =  52.998702

# ── Output directory ─────────────────────────────────────────────────────
BASE_DIR  = os.path.expanduser('~/Documents/FYP2026/nottingham900')
SCENE_DIR = os.path.join(BASE_DIR, 'scene')
MESH_DIR  = os.path.join(SCENE_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)

# ── Coordinate system ────────────────────────────────────────────────────
UTM_EPSG  = 32630   # UTM zone 30N (covers UK)

# ── AWS tile zoom level ──────────────────────────────────────────────────
# z=14 → ~2.4 m/px at UK latitude (512×512 px tiles)
# z=13 → ~4.8 m/px  (faster download, less detail)
TILE_ZOOM  = 14

# ── Terrain mesh resolution ───────────────────────────────────────────────
# Number of grid points per axis for the terrain PLY.
# 500 → 500×500 = 250k verts, ~500k triangles (~10 MB PLY) — recommended
# 200 → 200×200 = 40k  verts, ~80k triangles  (~1.5 MB PLY) — fast
TERRAIN_GRID_N = 500

# ── Building parameters ───────────────────────────────────────────────────
MIN_BUILDING_AREA_M2  = 30.0   # skip footprints smaller than this
CITY_MIN_HEIGHT_M     =  2.0   # clamp building height to at least this
CITY_MAX_HEIGHT_M     = 40.0   # clamp building height to at most this
HEIGHT_PER_LEVEL_M    =  3.5   # used when only building:levels tag present
DEFAULT_HEIGHT_M      =  8.0   # fallback when no height/levels tag

# ── OSM extra features ────────────────────────────────────────────────────
INCLUDE_ROADS       = True
INCLUDE_WATER       = True
INCLUDE_VEGETATION  = False   # polygon veg — adds clutter for macro-cell sim

# ── Exclude small/irrelevant building types ───────────────────────────────
EXCLUDE_BUILDING_TYPES = {
    'garage','garages','carport','shed','hut','roof','canopy',
    'kiosk','bicycle_parking','service','greenhouse','barn',
    'stable','sty','storage_tank','container','tent',
    'grandstand','shelter','utility','gatehouse',
}

print('Config loaded.')
print(f'  Scene bbox  : lon [{SCENE_WEST}, {SCENE_EAST}]')
print(f'                lat [{SCENE_SOUTH}, {SCENE_NORTH}]')
print(f'  Output      : {SCENE_DIR}')
print(f'  Tile zoom   : {TILE_ZOOM}  (~{156543.03 * np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2)) / 2**TILE_ZOOM:.1f} m/px)' if 'np' in dir() else f'  Tile zoom   : {TILE_ZOOM}')

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, time, json, struct, warnings
import numpy as np
import requests
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
import shapely.geometry as sg
import shapely.ops as so
from shapely.geometry import Polygon, MultiPolygon, box

# Optional: rasterio for GeoTIFF reading
try:
    import rasterio
    from rasterio.transform import from_bounds
    _HAS_RASTERIO = True
except ImportError:
    _HAS_RASTERIO = False
    print('⚠  rasterio not found — falling back to PIL for GeoTIFF tiles')

# PIL for fallback
try:
    from PIL import Image
    _HAS_PIL = True
except ImportError:
    _HAS_PIL = False

# osmnx for OSM data
try:
    import osmnx as ox
    ox.settings.use_cache = True
    ox.settings.log_console = False
    _HAS_OSMNX = True
except ImportError:
    _HAS_OSMNX = False
    print('⚠  osmnx not found — install with: pip install osmnx')

# trimesh for PLY export
try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
    print('⚠  trimesh not found — install with: pip install trimesh')

# Coordinate transformers
to_utm   = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
to_wgs84 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene bbox in UTM
sw_utm = to_utm.transform(SCENE_WEST,  SCENE_SOUTH)
ne_utm = to_utm.transform(SCENE_EAST,  SCENE_NORTH)
center_utm = ((sw_utm[0]+ne_utm[0])/2, (sw_utm[1]+ne_utm[1])/2)
center_lon, center_lat = to_wgs84.transform(*center_utm)

print(f'UTM SW : {sw_utm[0]:.1f}, {sw_utm[1]:.1f}')
print(f'UTM NE : {ne_utm[0]:.1f}, {ne_utm[1]:.1f}')
print(f'Center : ({center_lon:.5f}, {center_lat:.5f})')
print(f'Size   : {(ne_utm[0]-sw_utm[0])/1000:.2f} km × {(ne_utm[1]-sw_utm[1])/1000:.2f} km')

In [ ]:
# ============================================================
# CELL 2 — AWS ELEVATION TILES → HEIGHTMAP
# ============================================================
# Downloads GeoTIFF tiles from the public AWS elevation-tiles-prod bucket
# (same source used by Mapzen/Terrarium, Cesium, sionna-large-radio-maps).
# No credentials needed — anonymous public access.
# URL: s3://elevation-tiles-prod/geotiff/{z}/{x}/{y}.tif
# Values are direct metres ASL (float32 GeoTIFF, no conversion formula).

AWS_BASE = 'https://s3.amazonaws.com/elevation-tiles-prod/geotiff'

def _lon2tile(lon, z):
    return int(math.floor((lon + 180) / 360 * 2**z))

def _lat2tile(lat, z):
    lat_r = math.radians(lat)
    return int(math.floor((1 - math.log(math.tan(lat_r) + 1/math.cos(lat_r)) / math.pi) / 2 * 2**z))

def _tile2lon(x, z):
    return x / 2**z * 360 - 180

def _tile2lat(y, z):
    n = math.pi - 2 * math.pi * y / 2**z
    return math.degrees(math.atan(math.sinh(n)))

def _download_tile(z, x, y, retries=3):
    url = f'{AWS_BASE}/{z}/{x}/{y}.tif'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            buf = BytesIO(r.content)
            if _HAS_RASTERIO:
                with rasterio.open(buf) as ds:
                    data = ds.read(1).astype(np.float32)
            elif _HAS_PIL:
                # PIL may not read float GeoTIFF correctly — warn
                img = Image.open(buf)
                data = np.array(img, dtype=np.float32)
            else:
                raise RuntimeError('Need rasterio or PIL to read GeoTIFF tiles')
            # Resample to 512×512 if needed
            if data.shape != (512, 512):
                from scipy.ndimage import zoom as nd_zoom
                fx = 512 / data.shape[0]; fy = 512 / data.shape[1]
                data = nd_zoom(data, (fx, fy), order=1).astype(np.float32)
            return data
        except Exception as e:
            if attempt == retries - 1:
                print(f'  ⚠ tile {z}/{x}/{y} failed: {e} — using zeros')
                return np.zeros((512, 512), dtype=np.float32)
            time.sleep(2 ** attempt)

# Compute tile range for scene bbox
x0 = _lon2tile(SCENE_WEST,  TILE_ZOOM)
x1 = _lon2tile(SCENE_EAST,  TILE_ZOOM)
y0 = _lat2tile(SCENE_NORTH, TILE_ZOOM)  # north = smaller y in tile coords
y1 = _lat2tile(SCENE_SOUTH, TILE_ZOOM)
n_cols = x1 - x0 + 1
n_rows = y1 - y0 + 1
print(f'Tile range : x=[{x0},{x1}] y=[{y0},{y1}]  →  {n_cols}×{n_rows} = {n_cols*n_rows} tiles')

# Download in parallel
tile_mosaic = np.zeros((n_rows * 512, n_cols * 512), dtype=np.float32)
jobs = [(TILE_ZOOM, x0+col, y0+row, col, row)
        for row in range(n_rows) for col in range(n_cols)]

print(f'Downloading {len(jobs)} tiles ...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(_download_tile, z, x, y): (col, row)
               for z, x, y, col, row in jobs}
    for i, fut in enumerate(as_completed(futures)):
        col, row = futures[fut]
        tile_mosaic[row*512:(row+1)*512, col*512:(col+1)*512] = fut.result()
        if (i+1) % max(1, len(jobs)//4) == 0:
            print(f'  {i+1}/{len(jobs)} tiles done')

print(f'Download done in {time.time()-t0:.1f}s')

# Extent of the mosaic in WGS84
_map_min_lon = _tile2lon(x0,   TILE_ZOOM)
_map_max_lon = _tile2lon(x1+1, TILE_ZOOM)
_map_max_lat = _tile2lat(y0,   TILE_ZOOM)   # y0 is north
_map_min_lat = _tile2lat(y1+1, TILE_ZOOM)   # y1+1 is south
_mosaic_h, _mosaic_w = tile_mosaic.shape

print(f'Mosaic size : {_mosaic_w}×{_mosaic_h} px')
print(f'Mosaic lon  : [{_map_min_lon:.5f}, {_map_max_lon:.5f}]')
print(f'Mosaic lat  : [{_map_min_lat:.5f}, {_map_max_lat:.5f}]')
print(f'Elevation   : [{tile_mosaic.min():.1f}, {tile_mosaic.max():.1f}] m ASL')

def height_from_wgs84(lon, lat):
    """Bilinear interpolation from mosaic. Returns elevation in metres ASL."""
    u = (lon - _map_min_lon) / (_map_max_lon - _map_min_lon) * (_mosaic_w - 1)
    v = (1 - (lat - _map_min_lat) / (_map_max_lat - _map_min_lat)) * (_mosaic_h - 1)
    u = np.clip(u, 0, _mosaic_w - 1)
    v = np.clip(v, 0, _mosaic_h - 1)
    x0i, y0i = int(np.floor(u)), int(np.floor(v))
    x1i, y1i = min(x0i+1, _mosaic_w-1), min(y0i+1, _mosaic_h-1)
    fx, fy = u - x0i, v - y0i
    h = (tile_mosaic[y0i, x0i] * (1-fx) * (1-fy)
       + tile_mosaic[y0i, x1i] *    fx  * (1-fy)
       + tile_mosaic[y1i, x0i] * (1-fx) *    fy
       + tile_mosaic[y1i, x1i] *    fx  *    fy)
    return float(h)

def height_from_utm(easting, northing):
    """Height at UTM coordinates."""
    lon, lat = to_wgs84.transform(easting, northing)
    return height_from_wgs84(lon, lat)

# Scene centre elevation = local z=0 reference
origin_elev_asl = height_from_wgs84(center_lon, center_lat)
print(f'\nScene centre elevation : {origin_elev_asl:.2f} m ASL  (= local z=0)')

def local_z(lon, lat):
    """Returns local z in metres relative to scene centre elevation."""
    return height_from_wgs84(lon, lat) - origin_elev_asl

In [ ]:
# ============================================================
# CELL 3 — BUILD TERRAIN PLY
# ============================================================
# Generates a regular grid terrain mesh with DEM elevation.
# Building footprints are NOT cut out (no tool does this —
# buildings are separate meshes placed on the terrain surface).

N = TERRAIN_GRID_N
print(f'Building terrain mesh: {N}×{N} grid ({2*(N-1)**2:,} triangles) ...')

# UTM grid spanning scene bbox, centred at (0,0)
x_span = ne_utm[0] - sw_utm[0]
y_span = ne_utm[1] - sw_utm[1]
xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
XX, YY = np.meshgrid(xs, ys, indexing='ij')   # shape (N, N)

# DEM elevation for each grid point
ZZ = np.zeros((N, N), dtype=np.float32)
t0 = time.time()
for i in range(N):
    for j in range(N):
        utm_x = center_utm[0] + XX[i, j]
        utm_y = center_utm[1] + YY[i, j]
        ZZ[i, j] = height_from_utm(utm_x, utm_y) - origin_elev_asl
    if (i+1) % max(1, N//5) == 0:
        print(f'  row {i+1}/{N}  ({time.time()-t0:.0f}s)')

print(f'DEM range: [{ZZ.min():.1f}, {ZZ.max():.1f}] m (local)')

# Build vertex array and face array
verts = np.stack([XX.ravel(), YY.ravel(), ZZ.ravel()], axis=1)  # (N*N, 3)

faces = []
for i in range(N-1):
    for j in range(N-1):
        a = i*N + j
        b = a + 1
        c = a + N
        d = c + 1
        faces.append([a, b, d])
        faces.append([a, d, c])
faces = np.array(faces, dtype=np.int32)
print(f'Vertices: {len(verts):,}  Faces: {len(faces):,}')

# Export terrain PLY
terrain_ply = os.path.join(MESH_DIR, 'terrain.ply')
if _HAS_TRIMESH:
    mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
    mesh.export(terrain_ply)
else:
    # Minimal ASCII PLY writer
    with open(terrain_ply, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:
            f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for face in faces:
            f.write(f'3 {face[0]} {face[1]} {face[2]}\n')

sz = os.path.getsize(terrain_ply)/1024
print(f'Saved: {terrain_ply}  ({sz:.0f} KB)')

In [ ]:
# ============================================================
# CELL 4 — OSM BUILDINGS → BUILDING PLYS
# ============================================================
# Downloads OSM building footprints, determines correct base_z
# from the AWS heightmap (centroid query), extrudes walls + roof.

assert _HAS_OSMNX, 'osmnx required: pip install osmnx'

print('Downloading OSM buildings ...')
bbox_wgs = (SCENE_SOUTH, SCENE_NORTH, SCENE_WEST, SCENE_EAST)  # osmnx order
t0 = time.time()
try:
    gdf_bld = ox.features_from_bbox(
        north=SCENE_NORTH, south=SCENE_SOUTH,
        east=SCENE_EAST,   west=SCENE_WEST,
        tags={'building': True}
    )
except Exception as e:
    print(f'osmnx error: {e}')
    raise
print(f'  {len(gdf_bld)} raw building features  ({time.time()-t0:.1f}s)')

# ── Helper: wall material from OSM tags ────────────────────────────────────
def _bld_mat(row):
    mat  = str(row.get('building:material', '')).lower()
    fmat = str(row.get('building:facade:material', '')).lower()
    tag  = str(row.get('building', '')).lower()
    amen = str(row.get('amenity', '')).lower()
    shop = str(row.get('shop', '')).lower()
    off  = str(row.get('office', '')).lower()
    if 'glass' in mat or 'glass' in fmat:                return 'itu_glass'
    if 'wood'  in mat or 'timber' in mat:                return 'itu_wood'
    if 'wood'  in fmat or 'timber' in fmat:              return 'itu_wood'
    if 'brick' in mat or 'brick' in fmat:                return 'itu_brick'
    if 'stone' in mat or 'stone' in fmat:                return 'itu_brick'
    if tag in ('greenhouse','glasshouse'):               return 'itu_glass'
    if amen in ('shopping_centre','mall'):               return 'itu_glass'
    if shop in ('mall','supermarket','department_store'):return 'itu_glass'
    if off:                                              return 'itu_glass'
    if tag in ('residential','house','detached','semidetached_house',
               'semi_detached','terrace','terrace_house','bungalow',
               'farm','farmhouse','dormitory','apartments','block'):
        return 'itu_brick'
    if tag in ('industrial','warehouse','factory','shed',
               'storage_tank','silo','barn'):            return 'itu_concrete'
    if tag in ('retail','commercial','supermarket','kiosk'): return 'itu_glass'
    if tag in ('cathedral','church','chapel','mosque','temple'): return 'itu_brick'
    if tag in ('school','university','hospital','civic','public'): return 'itu_concrete'
    return 'itu_brick'   # UK default

def _roof_mat(row):
    roof_tag = str(row.get('roof:material', '')).lower()
    btag     = str(row.get('building', '')).lower()
    if any(k in roof_tag for k in ['metal','steel','zinc','aluminium','copper','tin']):
        return 'itu_metal'
    if 'glass' in roof_tag:
        return 'itu_glass'
    if any(k in roof_tag for k in ['wood','timber','thatch']):
        return 'itu_wood'
    if any(k in roof_tag for k in ['tile','concrete','slate','terracotta']):
        return 'itu_concrete'
    if btag in ('industrial','warehouse','factory','shed','barn',
                'retail','supermarket','commercial','garage','garages'):
        return 'itu_metal'
    return 'itu_concrete'   # UK residential default: concrete tile

def _bld_height(row):
    try:
        h = float(str(row.get('height','0')).replace('m','').strip())
        if h > 1:
            return np.clip(h, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)
    except:
        pass
    try:
        lvl = float(str(row.get('building:levels','0')).strip())
        if lvl > 0:
            return np.clip(lvl * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)
    except:
        pass
    return DEFAULT_HEIGHT_M

def _write_ply(verts, faces, path):
    """Write binary PLY."""
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(path)
        return
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:
            f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for fc in faces:
            f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')

def _extrude_building(poly_utm, base_z, height):
    """
    Extrude a 2D footprint polygon into wall + roof meshes (local coords).
    poly_utm : list of (x,y) in UTM, relative to scene centre
    base_z   : local z of ground at footprint centroid (metres)
    height   : building height above ground (metres)
    Returns (wall_verts, wall_faces, roof_verts, roof_faces)
    """
    pts = np.array(poly_utm, dtype=np.float32)
    if len(pts) < 3:
        return None
    # Remove duplicate closing vertex
    if np.allclose(pts[0], pts[-1]):
        pts = pts[:-1]
    n = len(pts)
    top_z    = base_z + height
    bot_z    = base_z

    # ── Walls ──────────────────────────────────────────────────────────────
    wall_v, wall_f = [], []
    for i in range(n):
        j = (i+1) % n
        # 4 vertices per wall quad
        v_base = len(wall_v)
        wall_v += [
            [pts[i,0], pts[i,1], bot_z],
            [pts[j,0], pts[j,1], bot_z],
            [pts[j,0], pts[j,1], top_z],
            [pts[i,0], pts[i,1], top_z],
        ]
        wall_f += [
            [v_base,   v_base+1, v_base+2],
            [v_base,   v_base+2, v_base+3],
        ]

    # ── Roof (flat, triangulated fan) ────────────────────────────────────
    roof_v = [[pts[i,0], pts[i,1], top_z] for i in range(n)]
    roof_cx = pts[:,0].mean()
    roof_cy = pts[:,1].mean()
    roof_v.append([roof_cx, roof_cy, top_z])
    centre_idx = len(roof_v) - 1
    roof_f = [[i, (i+1)%n, centre_idx] for i in range(n)]

    return (np.array(wall_v, np.float32),  np.array(wall_f,  np.int32),
            np.array(roof_v, np.float32),  np.array(roof_f,  np.int32))

# ── Process buildings ────────────────────────────────────────────────────
# Track PLY filenames per material for scene.xml
mat_plys = {}   # mat_name → list of ply filenames
n_ok = n_skip = 0

for idx, (oid, row) in enumerate(gdf_bld.iterrows()):
    geom = row.geometry
    if geom is None or geom.is_empty:
        n_skip += 1; continue

    # Use only exterior ring of Polygon (or largest Polygon of MultiPolygon)
    if isinstance(geom, MultiPolygon):
        geom = max(geom.geoms, key=lambda g: g.area)
    if not isinstance(geom, Polygon):
        n_skip += 1; continue

    # ── Skip excluded types ───────────────────────────────────────────────
    btag = str(row.get('building','')).lower()
    if btag in EXCLUDE_BUILDING_TYPES:
        n_skip += 1; continue

    # ── Convert footprint to UTM local coords ─────────────────────────────
    coords_wgs = list(geom.exterior.coords)
    coords_utm = []
    for lon, lat in coords_wgs:
        ex, ny = to_utm.transform(lon, lat)
        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))

    # ── Area filter ───────────────────────────────────────────────────────
    area_m2 = sg.Polygon(coords_utm).area
    if area_m2 < MIN_BUILDING_AREA_M2:
        n_skip += 1; continue

    # ── Building base z from heightmap at centroid ────────────────────────
    c_lon, c_lat = geom.centroid.x, geom.centroid.y
    base_z_local = local_z(c_lon, c_lat)

    # ── Height ────────────────────────────────────────────────────────────
    h = _bld_height(row)

    # ── Extrude ───────────────────────────────────────────────────────────
    result = _extrude_building(coords_utm, base_z_local, h)
    if result is None:
        n_skip += 1; continue
    wv, wf, rv, rf = result

    # ── Materials ─────────────────────────────────────────────────────────
    w_mat = _bld_mat(row)
    r_mat = _roof_mat(row)

    # ── Write PLYs ────────────────────────────────────────────────────────
    wall_name = f'bld_{n_ok:05d}_wall.ply'
    roof_name = f'bld_{n_ok:05d}_roof.ply'
    _write_ply(wv, wf, os.path.join(MESH_DIR, wall_name))
    _write_ply(rv, rf, os.path.join(MESH_DIR, roof_name))

    mat_plys.setdefault(w_mat, []).append(('meshes/' + wall_name, 'wall'))
    mat_plys.setdefault(r_mat, []).append(('meshes/' + roof_name, 'roof'))

    n_ok += 1
    if n_ok % 500 == 0:
        print(f'  {n_ok} buildings written ...')

print(f'\nBuildings : {n_ok} exported, {n_skip} skipped')
print(f'Materials : {list(mat_plys.keys())}')

In [ ]:
# ============================================================
# CELL 5 — WRITE SCENE.XML  (Mitsuba 2.1.0 / Sionna 0.19)
# ============================================================
# ITU-R P.2040-2 material definitions as used in Sionna 0.19.
# Each shape references a PLY file and a material by name.

ITU_MATERIALS = {
    # name          : (relative_permittivity, conductivity_S_m)
    'itu_concrete'  : (5.31,  0.092),
    'itu_brick'     : (3.75,  0.038),
    'itu_glass'     : (6.27,  0.000),
    'itu_wood'      : (1.99,  0.000),
    'itu_metal'     : (1.00, 1.0e7 ),
    'itu_asphalt'   : (2.56,  0.000),
    'itu_wet_ground': (30.0,  0.020),
}
TERRAIN_MATERIAL = 'itu_wet_ground'

# Collect all materials actually used
used_mats = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

lines = []
lines.append('<?xml version="1.0" encoding="utf-8"?>')
lines.append('<scene version="2.1.0">')
lines.append('')
lines.append('  <!-- ── ITU-R P.2040-2 Materials ────────────────────── -->')

for mat_name, (eps, sigma) in ITU_MATERIALS.items():
    if mat_name not in used_mats:
        continue
    lines.append(f'  <bsdf type="conductor" id="{mat_name}">')
    lines.append(f'    <float name="eta" value="{eps}"/>')
    lines.append(f'    <float name="k"   value="{sigma}"/>')
    lines.append(f'  </bsdf>')
    lines.append('')

lines.append('  <!-- ── Terrain ─────────────────────────────────────── -->')
lines.append('  <shape type="ply">')
lines.append('    <string name="filename" value="meshes/terrain.ply"/>')
lines.append('    <ref id="{mat}" name="bsdf"/>'.replace('{mat}', TERRAIN_MATERIAL))
lines.append('  </shape>')
lines.append('')

lines.append('  <!-- ── Buildings ───────────────────────────────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    for ply_path, role in ply_list:
        lines.append(f'  <shape type="ply">')
        lines.append(f'    <string name="filename" value="{ply_path}"/>')
        lines.append(f'    <ref id="{mat_name}" name="bsdf"/>')
        lines.append(f'  </shape>')

lines.append('')
lines.append('</scene>')

scene_xml = os.path.join(SCENE_DIR, 'scene.xml')
with open(scene_xml, 'w') as f:
    f.write('\n'.join(lines))

print(f'Wrote: {scene_xml}')
print(f'  Materials : {len(used_mats)}')
total_shapes = 1 + sum(len(v) for v in mat_plys.values())
print(f'  Shapes    : {total_shapes}  (1 terrain + {total_shapes-1} building parts)')

# Save scene metadata for main notebook
meta = {
    'scene_center_lon'  : center_lon,
    'scene_center_lat'  : center_lat,
    'origin_elev_asl_m' : origin_elev_asl,
    'utm_epsg'          : UTM_EPSG,
    'bbox'              : {'west': SCENE_WEST, 'east': SCENE_EAST,
                           'south': SCENE_SOUTH, 'north': SCENE_NORTH},
    'n_buildings'       : n_ok,
    'terrain_grid_n'    : TERRAIN_GRID_N,
    'tile_zoom'         : TILE_ZOOM,
}
params_json = os.path.join(BASE_DIR, 'scene_parameters.json')
with open(params_json, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Metadata  : {params_json}')

In [ ]:
# ============================================================
# CELL 6 — VERIFY SCENE
# ============================================================
# Quick sanity checks before handing the scene to the main notebook.

import glob as glob_mod

ply_files = sorted(glob_mod.glob(os.path.join(MESH_DIR, '*.ply')))
total_kb = sum(os.path.getsize(p) for p in ply_files) / 1024

print('=' * 60)
print('SCENE VERIFICATION')
print('=' * 60)
print(f'PLY files   : {len(ply_files)}')
print(f'Total size  : {total_kb/1024:.1f} MB')
print()

print(f'scene.xml   : {os.path.getsize(scene_xml)/1024:.0f} KB')

# Load with Mitsuba to verify (requires Sionna env)
try:
    import mitsuba as mi
    mi.set_variant('scalar_rgb')
    scene_mi = mi.load_file(scene_xml)
    bbox = scene_mi.bbox()
    print()
    print(f'Mitsuba load: OK')
    print(f'  BBox X    : [{float(bbox.min[0]):.1f}, {float(bbox.max[0]):.1f}] m')
    print(f'  BBox Y    : [{float(bbox.min[1]):.1f}, {float(bbox.max[1]):.1f}] m')
    print(f'  BBox Z    : [{float(bbox.min[2]):.1f}, {float(bbox.max[2]):.1f}] m')
except Exception as e:
    print(f'Mitsuba load: {e}')

print()
print('Scene metadata (scene_parameters.json):')
with open(params_json) as f:
    print(json.dumps(json.load(f), indent=2))

print()
print('DONE — scene ready for sionna019_main_simulation.ipynb')
print(f'Set BASE_DIR = "{BASE_DIR}" in Cell 0c of the main notebook.')